# ベースライン学習（検証: 時系列CV）

- **使う特徴量**: 34（3C3）＋ テキストメタ 4 列 ＝ **38 特徴**（movie_info_and_title_meta。train_text_experiments で CV 最良だった設定）。
- **検証**: 時系列CV（2013〜2016 を val）。提出は全 train で 1 本学習して test を予測。

> 改善実験は `train_metric_driven_experiments.ipynb` を使用。

In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from lib import get_baseline_data, BASELINE_FEATURES, add_prediction_analysis, summarize_errors_by, run_full_analysis

OUTPUT_DIR = "outputs"
os.makedirs(os.path.join(OUTPUT_DIR, "submissions"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "analysis"), exist_ok=True)
print("Setup complete.")

Setup complete.


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [3]:
# データ読み込み〜特徴量構築まで lib で一括実行（38特徴）
train, test = get_baseline_data()
FEATURES = BASELINE_FEATURES
print(f"Train: {len(train):,}, Test: {len(test):,}, Features: {len(FEATURES)}")

Train: 653,507, Test: 40,716, Features: 38


In [4]:
train.head()

,ID,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_date,movie_title,movie_info,content_rating,genres,...,production_company_te_ts,critic_name_te_ts_bin,production_company_te_ts_bin,runtime_bin,movie_age_bin,release_decade,movie_info_len,movie_info_word_count,movie_title_len,movie_title_word_count
0,0,m/0814255,Andrew L. Urban,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489462,2.0,1.0,1.0,0,2010.0,454,79,50,8
1,1,m/0814255,Louise Keller,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489491,2.0,1.0,1.0,0,2010.0,454,79,50,8
2,2,m/0814255,David Germain,True,Associated Press,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489582,1.0,1.0,1.0,0,2010.0,454,79,50,8
3,3,m/0814255,Nick Schager,False,Slant Magazine,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489637,1.0,1.0,1.0,0,2010.0,454,79,50,8
4,4,m/0814255,Bill Goodykoontz,True,Arizona Republic,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0.489549,2.0,1.0,1.0,0,2010.0,454,79,50,8


In [5]:
# 時系列CV: 検証年ごとに「その年より前で学習 → その年で検証」
# train の review_year 範囲に合わせて検証年を設定（test が 2017〜 なので 2013〜2016 などで検証）
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))

print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")
for i, (_, val_idx) in enumerate(time_splits):
    print(f"  Fold{i+1}: val_year={VAL_YEARS[i]}, val_n={len(val_idx):,}")

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])
  Fold1: val_year=2013, val_n=47,263
  Fold2: val_year=2014, val_n=45,165
  Fold3: val_year=2015, val_n=49,842
  Fold4: val_year=2016, val_n=36,455


In [7]:
X = train[FEATURES]
y = train["target"].values
X_test = test[FEATURES]

lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
    "random_state": 42, "verbosity": -1,
}

oof = np.zeros(len(train))
test_preds = []
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(time_splits):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    fold_scores.append(fold_auc)
    test_preds.append(model.predict_proba(X_test)[:, 1])
    print(f"Fold{fold+1}: val_year={VAL_YEARS[fold]}, AUC={fold_auc:.4f}")

# 時系列CVでは oof が全行を埋めないため、CV AUC は fold 平均で表示
print(f"\nCV AUC (fold mean): {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

Fold1: val_year=2013, AUC=0.7602
Fold2: val_year=2014, AUC=0.7660
Fold3: val_year=2015, AUC=0.7512
Fold4: val_year=2016, AUC=0.7634

CV AUC (fold mean): 0.7602 ± 0.0056


In [8]:
# 提出用: 全 train で 1 本学習して test を予測（ベースラインと同じく「全データで学習→予測」）
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X, y)
final_pred = model_full.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"ID": test["ID"], "target": final_pred})
out_path = os.path.join(OUTPUT_DIR, "submissions", "submission.csv")
submission.to_csv(out_path, index=False)
print(f"Saved {out_path} (rows: {len(submission):,}) [全trainで学習した1本のモデルで予測]")

# 特徴量重要度（model_full の gain）
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_full.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance_df)
print(f"\n重要度 Top5: {list(importance_df.head(5)['feature'].values)}")

Saved submission.csv (rows: 40,716) [全trainで学習した1本のモデルで予測]


,feature,importance
7,directors,754
8,authors,550
0,rotten_tomatoes_link,453
4,movie_title,420
1,critic_name,228
3,publisher_name,172
9,actors,96
11,production_company,86
27,critic_name_te_ts,51
5,movie_info,45



重要度 Top5: ['directors', 'authors', 'rotten_tomatoes_link', 'movie_title', 'critic_name']


In [9]:
# 予測の当たり・外れ分析（CV の oof を使い、どのセグメントで外しているかを集計）
# oof は時系列CVで val 部分のみ埋まるので、val に含まれる行だけ分析対象にする
val_mask = np.zeros(len(train), dtype=bool)
for _tr_idx, val_idx in time_splits:
    val_mask[val_idx] = True
train_val = train.loc[val_mask].copy()
oof_val = oof[val_mask]
y_val = train["target"].values[val_mask]
analysis_df, summaries = run_full_analysis(
    train_val, y_val, oof_val,
    group_cols=["review_year", "content_rating", "genre_Drama", "genre_Documentary", "top_critic"],
    min_count=100,
)
# 分析結果の保存（改善のヒント用）
analysis_df.to_csv(os.path.join(OUTPUT_DIR, "analysis", "train_with_predictions.csv"), index=False)
print(f"Saved {OUTPUT_DIR}/analysis/train_with_predictions.csv (target, pred, outcome=TP/TN/FP/FN 付き)")

for col, summary in summaries.items():
    fname = os.path.join(OUTPUT_DIR, "analysis", f"prediction_summary_by_{col}.csv")
    summary.to_csv(fname, index=False)
    print(f"  {fname}: n={len(summary)}")
    # AUC が低い or FN/FP 率が高いセグメントを表示
    if "auc" in summary.columns and summary["auc"].notna().any():
        worst = summary.loc[summary["auc"].idxmin()]
        print(f"    最低AUC: {worst[col]} (AUC={worst['auc']:.4f}, n={worst['n']})")
print("\nセグメント別集計の例 (review_year):")
display(summaries.get("review_year", pd.DataFrame()))

Saved train_with_predictions.csv (target, pred, outcome=TP/TN/FP/FN 付き)
  prediction_summary_by_review_year.csv: n=4
    最低AUC: 2015.0 (AUC=0.7512, n=49842.0)
  prediction_summary_by_content_rating.csv: n=6
    最低AUC: NC17 (AUC=0.7312, n=240)
  prediction_summary_by_genre_Drama.csv: n=2
    最低AUC: 1.0 (AUC=0.7548, n=94252.0)
  prediction_summary_by_genre_Documentary.csv: n=2
    最低AUC: 1.0 (AUC=0.7149, n=14931.0)
  prediction_summary_by_top_critic.csv: n=2
    最低AUC: True (AUC=0.7467, n=54956)

セグメント別集計の例 (review_year):


,review_year,n,n_TP,n_TN,n_FP,n_FN,accuracy,auc,FP_rate,FN_rate,pos_rate,pred_pos_rate
0,2013,47263,26361,7719,9000,4183,0.721071,0.760210,0.538310,0.136950,0.646256,0.748175
1,2014,45165,25122,7354,8838,3851,0.719052,0.766005,0.545825,0.132917,0.641492,0.751910
2,2015,49842,28569,7266,9557,4450,0.718972,0.751233,0.568091,0.134771,0.662473,0.764937
3,2016,36455,20959,5319,7062,3115,0.720834,0.763433,0.570390,0.129393,0.660376,0.768646
